# chain-rule-elementwise — ex1: write sigmoid_back and relu_back from the elementwise chain rule

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `chain-rule-elementwise`. Running the final beacon cell reports progress against the `Backprop: Elementwise chain rule` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """A minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries an optional `.recipe`
    populated by wrap_forward_fn. `requires_grad` is set by the wrapper."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: Elementwise chain rule` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`chain-rule-elementwise`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "chain-rule-elementwise"
DD_SUBTOPIC = "Backprop: Elementwise chain rule"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Elementwise chain rule — quick refresher

For an elementwise forward op `out = f(x)` (so `out[i] = f(x[i])` independently per index), the local Jacobian is **diagonal**: `d(out[i]) / d(x[j])` is `f'(x[i])` when `i == j` and `0` otherwise.

The chain rule then collapses to a per-position product:

```
dL/dx[i] = sum_j  dL/dout[j] * d(out[j])/d(x[i])
         = dL/dout[i] * f'(x[i])     # only diagonal term survives
```

So elementwise backward fns are one-liners — multiply `grad_out` by the elementwise local derivative, no actual Jacobian matrix is materialized. Shape of `grad_in` always equals shape of `x`.

### Exercise 1 — write sigmoid_back and relu_back from the elementwise chain rule

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Bloom level: Apply
> LO: Apply the elementwise chain rule to derive sigmoid_back and relu_back, returning grad_in = grad_out * f'(x) without materializing any Jacobian matrix.
> Keywords: chain-rule, sigmoid, relu, elementwise-derivative
> ```

**KCs targeted:** `chain-rule-elementwise`, `back-fn-uses-cached-out`

Implement TWO elementwise backward fns by working out the local derivative and multiplying it by `grad_out`:

**1. `sigmoid_back(grad_out, out, x)`** — gradient of `out = 1 / (1 + exp(-x))`.
   - The clean form uses `out`: `d/dx sigmoid(x) = sigmoid(x) * (1 - sigmoid(x)) = out * (1 - out)`.
   - So `dL/dx = grad_out * out * (1 - out)`.

**2. `relu_back(grad_out, out, x)`** — gradient of `out = max(x, 0)`.
   - The local derivative is `1` where `x > 0`, `0` elsewhere (undefined at exactly 0; convention: use 0).
   - So `dL/dx = grad_out * (x > 0)`. (Multiplying by a bool tensor is fine — PyTorch promotes it to the float dtype.)

**The point.** Both ops are elementwise, so the Jacobian is diagonal, so the chain rule reduces to a per-position product. No matrix is materialized — `grad_in.shape == x.shape` always.

Inputs are plain `torch.Tensor`; no autograd. Return tensors with the same shape and float dtype as `x`.

In [ ]:
def sigmoid_back(grad_out: Tensor, out: Tensor, x: Tensor) -> Tensor:
    # d/dx sigmoid(x) = sigmoid(x) * (1 - sigmoid(x)) = out * (1 - out).
    # Using `out` is faster (no second exp call) and numerically stabler.
    return grad_out * out * (1 - out)


def relu_back(grad_out: Tensor, out: Tensor, x: Tensor) -> Tensor:
    # d/dx relu(x) = 1 for x > 0 else 0. Multiply grad_out by the mask.
    # Bool * float in torch promotes to float — no manual cast needed.
    return grad_out * (x > 0)


<details><summary>Solution</summary>

```python
def sigmoid_back(grad_out: Tensor, out: Tensor, x: Tensor) -> Tensor:
    # d/dx sigmoid(x) = sigmoid(x) * (1 - sigmoid(x)) = out * (1 - out).
    # Using `out` is faster (no second exp call) and numerically stabler.
    return grad_out * out * (1 - out)


def relu_back(grad_out: Tensor, out: Tensor, x: Tensor) -> Tensor:
    # d/dx relu(x) = 1 for x > 0 else 0. Multiply grad_out by the mask.
    # Bool * float in torch promotes to float — no manual cast needed.
    return grad_out * (x > 0)
```

**Why these are one-liners.** Elementwise ops have a diagonal local Jacobian. The chain-rule sum `dL/dx[i] = sum_j (dL/dout[j] * d(out[j])/d(x[i]))` collapses to `dL/dout[i] * f'(x[i])` because only the diagonal term survives. No matrix is materialized — that's why elementwise back fns are O(n) instead of O(n^2).

**`out * (1 - out)` vs recomputing.** You could write `sig = t.sigmoid(x); return grad_out * sig * (1 - sig)`, but the whole reason `out` is in the back-fn signature is so you DON'T have to. Saves one exp per node on the reverse pass.

**Why `relu_back` ignores `out`.** `out` is `max(x, 0)`, which doesn't tell you whether `x > 0` (when `x = 0`, `out = 0` either way). You have to read `x` directly. This is why the uniform signature passes BOTH `out` AND `x` — different ops need different cached state.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()